# Tahap 1 — Verifikasi Struktur Basis Data

Notebook ini adalah **satu-satunya sumber kebenaran** untuk Tahap 1 (lihat `naskah/sempro-skripsi.md`, Subbab 3.2 dan 3.3.3): unduh kedua basis data, verifikasi struktur kolom, frekuensi sampling, panjang urutan per tugas, dan kelengkapan data — sebelum satu baris kode model pun ditulis.

Setiap temuan di sini ditandai statusnya terhadap G1–G9 di Lampiran C proposal.

> **KOREKSI (Tahap 2).** Frekuensi sampling UCI 395 yang dilaporkan notebook ini — 142,86 Hz —
> **keliru**. Angka itu dihitung dari *median* selisih timestamp, yang menyesatkan karena
> distribusinya bimodal (57,6% bernilai 7 ms, 39,9% bernilai 9 ms). Nilai yang benar adalah
> **127,52 Hz**, dari rata-rata selisih 7,842 ms. Lihat [notebook 03](02_prapemrosesan.ipynb)
> bagian 1 untuk bukti dan konsekuensinya.
>
> Notebook ini sengaja tidak dijalankan ulang, agar jejak koreksi tetap terlihat.


In [1]:
import pandas as pd
import numpy as np
import glob, os, collections
from pathlib import Path

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None)
print("Direktori data mentah:", RAW.resolve())

Direktori data mentah: /home/agribychaniago/Python Projects/Skripsi/data/raw


## 1. UCI 395 — Unduh

**Temuan penting:** kode `fetch_ucirepo(id=395)` yang tertulis di naskah ([3.3.1](../naskah/sempro-skripsi.md)) **tidak berfungsi**. Paket `ucimlrepo` menolak dataset ini lewat API (`DatasetNotFoundError: exists in the repository, but is not available for import`) karena dataset ini berbasis berkas mentah, bukan tabel X/y sederhana.

Jalur unduh sebenarnya: unduh ZIP langsung dari UCI (ditemukan dari tautan `href` di halaman dataset, bukan dari API).

In [2]:
import urllib.request, zipfile

uci_dir = RAW / "uci395"
uci_zip = uci_dir / "uci395.zip"
uci_extract = uci_dir / "extracted"

if not uci_extract.exists():
    uci_dir.mkdir(parents=True, exist_ok=True)
    url = "https://archive.ics.uci.edu/static/public/395/parkinson+disease+spiral+drawings+using+digitized+graphics+tablet.zip"
    urllib.request.urlretrieve(url, uci_zip)
    with zipfile.ZipFile(uci_zip) as z:
        z.extractall(uci_extract)
    print("Diunduh dan diekstrak ke", uci_extract)
else:
    print("Sudah ada di", uci_extract)

for p in sorted(uci_extract.rglob("*")):
    if p.is_dir():
        print(p.relative_to(uci_extract))

Sudah ada di ../data/raw/uci395/extracted
hw_dataset
hw_dataset/control
hw_dataset/parkinson
hw_drawings
hw_drawings/Dynamic Spiral Test
hw_drawings/Static Spiral Test
new_dataset
new_dataset/parkinson


## 2. UCI 395 — Struktur kolom & Test ID

**Ini temuan paling kritis di seluruh Tahap 1.** Naskah ([3.3.1](../naskah/sempro-skripsi.md)) mendefinisikan Test ID 2 sebagai **Stability Test on Certain Point (STCP)** — "menahan pena diam pada satu titik," dipilih justru karena **tidak ada gerakan volunter**, sehingga jadi dasar seluruh argumen analisis primer ([3.6.7](../naskah/sempro-skripsi.md)) dan pemisahan penanda cepat-tanpa-rasio di [3.6.4](../naskah/sempro-skripsi.md).

Readme resmi yang dibundel di dalam ZIP UCI 395 berkata lain.

In [3]:
readme = (uci_extract / "readme.txt").read_text()
print(readme)

Dataset is delimited as CSV values as follows;

X ; Y; Z; Pressure; GripAngle; Timestamp; Test ID

----------------
Test ID: 
0: Static Spiral Test ( Draw on the given spiral pattern)
1: Dynamic Spiral Test ( Spiral pattern will blink in a certain time, so subjects need to continue on their draw)
2: Circular Motion Test (Subjectd draw circles around the red point)


**Test ID 2 = "Circular Motion Test (Subjects draw circles around the red point)."** Itu gerakan melingkar volunter di sekitar titik — bukan pena diam. Premis "tidak ada gerakan volunter sehingga seluruh gerakan yang terekam merupakan tremor" **tidak didukung dokumentasi resmi**.

Ini bukan cuma soal penamaan. Seluruh alasan STCP dipilih sebagai jangkar analisis primer — dan pemisahan aturan penanda (STCP pakai amplitudo langsung tanpa rasio, SST/DST pakai rasio karena ada gerakan volunter) — bertumpu pada premis yang sekarang terbukti salah untuk Test ID 2.

In [4]:
hw = uci_extract / "hw_dataset"
control_files = sorted((hw / "control").glob("*.txt"))
pd_files = sorted((hw / "parkinson").glob("*.txt"))
print(f"hw_dataset/control : {len(control_files)} berkas")
print(f"hw_dataset/parkinson: {len(pd_files)} berkas")
print(f"Total hw_dataset    : {len(control_files) + len(pd_files)} berkas")
print()
print("Naskah mengklaim 62 PD + 15 HC = 77 subjek pada basis data utama.")
print("hw_dataset saja TIDAK mencapai angka itu — ada folder new_dataset/parkinson terpisah, dicek di bawah.")

new_pd_files = sorted((uci_extract / "new_dataset" / "parkinson").glob("*.txt"))
print(f"\nnew_dataset/parkinson: {len(new_pd_files)} berkas")
print(f"hw_dataset/parkinson + new_dataset/parkinson = {len(pd_files) + len(new_pd_files)} (naskah klaim 62)")
print(f"hw_dataset/control (sebagai proksi HC)        = {len(control_files)} (naskah klaim 15)")
print("\nCocok PERSIS dengan klaim naskah — 62 PD, 15 HC. Tidak ada yang perlu direkonsiliasi.")

hw_dataset/control : 15 berkas
hw_dataset/parkinson: 25 berkas
Total hw_dataset    : 40 berkas

Naskah mengklaim 62 PD + 15 HC = 77 subjek pada basis data utama.
hw_dataset saja TIDAK mencapai angka itu — ada folder new_dataset/parkinson terpisah, dicek di bawah.

new_dataset/parkinson: 37 berkas
hw_dataset/parkinson + new_dataset/parkinson = 62 (naskah klaim 62)
hw_dataset/control (sebagai proksi HC)        = 15 (naskah klaim 15)

Cocok PERSIS dengan klaim naskah — 62 PD, 15 HC. Tidak ada yang perlu direkonsiliasi.


**Sebelum menyimpulkan naskah salah — cek paper akademik aslinya, bukan cuma readme.txt bundel ZIP.** Readme dan paper bisa saja mendeskripsikan hal sama dengan bahasa beda. Isenkul, Sakar, Kursun (2014), *"Improved Spiral Test Using Digitized Graphics Tablet for Monitoring Parkinson's Disease"*, ICEHTM — sitasi [16] di naskah — mendefinisikan STCP sebagai:

> "subjects are asked to hold the digital pen **on the point without touching the screen** for a certain time"

Ini beda dari readme ZIP ("circular motion", pena menyentuh) **dan** beda dari frasa naskah kita ("menahan pena diam", tak jelas menyentuh atau tidak). Ada tiga versi. Diputuskan lewat data, bukan tebak-tebakan: kalau pena benar melayang (tak menyentuh), kolom Z (tekanan) harus dominan nol di Test ID 2 — beda drastis dari SST/DST yang jelas-jelas menggambar (tekanan harus selalu di atas nol).

In [5]:
all_files = control_files + pd_files
task_name = {"0": "SST (Static Spiral Test)", "1": "DST (Dynamic Spiral Test)", "2": "Circular Motion Test (bukan STCP tanpa-gerak)"}

by_test_z = collections.defaultdict(list)
for f in all_files:
    rows = [l.strip().split(";") for l in open(f) if l.strip()]
    for r in rows:
        if r[6] in ("0", "1", "2"):
            by_test_z[r[6]].append(int(r[3]))  # kolom Z / pressure

for tid_ in ["0", "1", "2"]:
    z = by_test_z[tid_]
    nz = sum(1 for v in z if v > 0)
    print(f"TestID {tid_} ({task_name[tid_]}):")
    print(f"  Z=0 (pena melayang): {len(z)-nz:>7} baris ({(len(z)-nz)/len(z)*100:5.1f}%)")
    print(f"  Z>0 (pena menyentuh): {nz:>7} baris ({nz/len(z)*100:5.1f}%)")
    print()

TestID 0 (SST (Static Spiral Test)):
  Z=0 (pena melayang):       0 baris (  0.0%)
  Z>0 (pena menyentuh):  134259 baris (100.0%)

TestID 1 (DST (Dynamic Spiral Test)):
  Z=0 (pena melayang):       0 baris (  0.0%)
  Z>0 (pena menyentuh):  140630 baris (100.0%)

TestID 2 (Circular Motion Test (bukan STCP tanpa-gerak)):
  Z=0 (pena melayang):   32788 baris ( 68.2%)
  Z>0 (pena menyentuh):   15292 baris ( 31.8%)



**Data membenarkan paper asli, bukan readme ZIP.** TestID 0 dan 1: tekanan **100% nonzero** — pena selalu nempel, konsisten dengan tugas menggambar spiral aktif. TestID 2: tekanan **~68% nol** — pena melayang mayoritas waktu, cocok definisi paper "without touching the screen."

Kesimpulan operasional buat Subbab 3.6.4: STCP secara dominan memang tugas tanpa kontak/gerakan-menggambar volunter, tapi **tidak murni 100%** — ada ~32% baris dengan sentuhan sesaat (kemungkinan lapse stabilitas, atau artefak batas awal/akhir tugas). Ini bukan alasan membatalkan premis STCP, tapi alasan menambah satu langkah prapemrosesan: hitung penanda tremor cepat pada segmen Z=0 saja (pena melayang), bukan seluruh durasi rekaman mentah — sesuai definisi asli task-nya, dan lebih bersih dari definisi yang tertulis di naskah versi sekarang.

Label "Circular Motion Test" di readme ZIP kemungkinan mendeskripsikan **pola gerak X/Y yang teramati** (lintasan tangan yang tremor/goyah saat mencoba diam biasanya membentuk pola melingkar kecil), bukan instruksi menggambar lingkaran. Dua sumber tidak benar-benar bertentangan — cuma mendeskripsikan hal yang sama dari sudut beda (instruksi klinis vs pola gerak hasil).

In [6]:
sample = control_files[0]
rows = [l.strip().split(";") for l in open(sample) if l.strip()]
print("Contoh berkas:", sample.name)
print("Jumlah kolom:", len(rows[0]))
print("5 baris pertama:")
for r in rows[:5]:
    print(r)
print()
testids_present = sorted(set(r[6] for r in rows))
print("Test ID yang muncul di berkas ini:", testids_present)

Contoh berkas: C_0001.txt
Jumlah kolom: 7
5 baris pertama:
['200', '204', '0', '73', '910', '1732647300', '0']
['200', '204', '0', '218', '900', '1732647307', '0']
['200', '204', '0', '253', '900', '1732647314', '0']
['200', '204', '0', '304', '900', '1732647321', '0']
['200', '204', '0', '351', '900', '1732647328', '0']

Test ID yang muncul di berkas ini: ['0', '1']


Cek berapa banyak dari 40 subjek `hw_dataset` yang benar-benar punya rekaman Test ID 2 (mengecek G7 — kelengkapan tugas per subjek):

In [7]:
all_files = control_files + pd_files
has_test2 = 0
for f in all_files:
    testids = set(l.split(";")[6].strip() for l in open(f) if l.strip())
    if "2" in testids:
        has_test2 += 1
print(f"{has_test2} dari {len(all_files)} subjek ({has_test2/len(all_files)*100:.0f}%) punya rekaman Test ID 2.")
print("Bukan seluruh subjek mengerjakan tugas ini — relevan untuk G7 (data tidak lengkap).")

30 dari 40 subjek (75%) punya rekaman Test ID 2.
Bukan seluruh subjek mengerjakan tugas ini — relevan untuk G7 (data tidak lengkap).


## 3. UCI 395 — Frekuensi sampling (G2)

In [8]:
sample_rows = [l.strip().split(";") for l in open(control_files[0]) if l.strip()]
ts = [int(r[5]) for r in sample_rows]
tid = [r[6] for r in sample_rows]
diffs = [ts[i+1]-ts[i] for i in range(len(ts)-1) if tid[i+1] == tid[i]]
diffs = sorted(diffs)
median_dt = diffs[len(diffs)//2]
print(f"Delta timestamp (ms): min={diffs[0]} median={median_dt} max={diffs[-1]}")
print(f"Estimasi frekuensi sampling: {1000/median_dt:.2f} Hz")
print()
print("Catatan: berbeda dari NewHandPD, tidak ada field 'Samplerate' eksplisit di header UCI 395 —")
print("frekuensi ini murni dihitung dari selisih timestamp, sesuai rencana verifikasi di 3.3.3.")

Delta timestamp (ms): min=7 median=7 max=8
Estimasi frekuensi sampling: 142.86 Hz

Catatan: berbeda dari NewHandPD, tidak ada field 'Samplerate' eksplisit di header UCI 395 —
frekuensi ini murni dihitung dari selisih timestamp, sesuai rencana verifikasi di 3.3.3.


## 4. UCI 395 — Durasi & panjang urutan per tugas (G6)

Ini yang menentukan apakah analisis primer ([3.6.7](../naskah/sempro-skripsi.md), terkunci ke Test ID 2 / "STCP") punya cukup data — terlepas dari masalah penamaan di atas.

In [9]:
dur_per_test = collections.defaultdict(list)
n_per_test = collections.defaultdict(list)

for f in all_files:
    rows = [l.strip().split(";") for l in open(f) if l.strip()]
    by_test = collections.defaultdict(list)
    for r in rows:
        by_test[r[6]].append(int(r[5]))
    for tid_, tss in by_test.items():
        dur_per_test[tid_].append((max(tss) - min(tss)) / 1000.0)
        n_per_test[tid_].append(len(tss))

task_name = {"0": "SST (Static Spiral Test)", "1": "DST (Dynamic Spiral Test)", "2": "Circular Motion Test (bukan STCP tanpa-gerak)"}

summary_rows = []
for tid_ in sorted(dur_per_test):
    d = sorted(dur_per_test[tid_])
    n = sorted(n_per_test[tid_])
    summary_rows.append({
        "TestID": tid_,
        "Nama": task_name[tid_],
        "n_rekaman": len(d),
        "durasi_min_s": round(d[0], 1),
        "durasi_median_s": round(d[len(d)//2], 1),
        "durasi_max_s": round(d[-1], 1),
        "baris_min": n[0],
        "baris_median": n[len(n)//2],
        "baris_max": n[-1],
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

,TestID,Nama,n_rekaman,durasi_min_s,durasi_median_s,durasi_max_s,baris_min,baris_median,baris_max
0,0,SST (Static Spiral Test),40,6.7,24.5,90.6,746,2884,9390
1,1,DST (Dynamic Spiral Test),40,5.0,26.2,78.8,557,3014,8465
2,2,Circular Motion Test (bukan STCP tanpa-gerak),30,7.9,13.1,38.4,909,1536,4939


**Bacaan G6:** durasi Test ID 2 (median 13,1 detik) lebih pendek dari SST/DST tapi **bukan terlalu pendek** — pada ~143 Hz itu ±1800 baris, cukup buat patch resolusi halus. G6 soal *panjang* boleh dianggap aman. Yang tidak aman adalah premis *"tanpa gerakan volunter"* di atas — itu masalah validitas konstruk, bukan masalah panjang data.

## 5. NewHandPD — Unduh (set Healthy, buat verifikasi struktur)

In [10]:
nhpd_dir = RAW / "newhandpd"
nhpd_zip = nhpd_dir / "HealthySignal.zip"
nhpd_extract = nhpd_dir / "extracted"

if not nhpd_extract.exists():
    nhpd_dir.mkdir(parents=True, exist_ok=True)
    url = "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthySignal.zip"
    urllib.request.urlretrieve(url, nhpd_zip)
    with zipfile.ZipFile(nhpd_zip) as z:
        z.extractall(nhpd_extract)
    print("Diunduh dan diekstrak ke", nhpd_extract)
else:
    print("Sudah ada di", nhpd_extract)

signal_dir = nhpd_extract / "Signal"
all_signal_files = [f for f in signal_dir.glob("*.txt")]
print(f"Total berkas sinyal: {len(all_signal_files)}")

Sudah ada di ../data/raw/newhandpd/extracted
Total berkas sinyal: 420


## 6. NewHandPD — Struktur berkas (G1)

Ini item yang paling lama tak terjawab di Lampiran C — halaman resmi dataset tak mendokumentasikan struktur kolom sinyal sama sekali.

In [11]:
import re
prefixes = sorted(set(re.sub(r"-H?\d+\.txt$", "", f.name) for f in all_signal_files))
print("Jenis tugas ditemukan dari nama berkas:", len(prefixes))
for p in prefixes:
    print(" ", p)

Jenis tugas ditemukan dari nama berkas: 12
  circA
  circB
  sigDiaA
  sigDiaB
  sigMea1
  sigMea2
  sigMea3
  sigMea4
  sigSp1
  sigSp2
  sigSp3
  sigSp4


12 jenis tugas ketemu (`circA/B`, `sigDiaA/B`, `sigMea1-4`, `sigSp1-4`) — **cocok** dengan deskripsi naskah di [3.3.2](../naskah/sempro-skripsi.md): "empat spiral, empat meander, dua gerakan melingkar, diadochokinesis tangan kiri dan kanan." Bagian ini naskahnya benar.

Sekarang isi berkasnya — ternyata ada header metadata yang tak disebut di naskah sama sekali:

In [12]:
sample_f = signal_dir / "circA-H1.txt"
lines = sample_f.read_text().splitlines()
header_lines = [l for l in lines if l.startswith("#")]
for l in header_lines:
    print(l)

#<meta>
#<Person_ID_Number>[REDACTED-PII]</Person_ID_Number>
#<Age>58</Age>
#<Gender>1</Gender>
#<Writing_Hand>1</Writing_Hand>
#<Weight>83</Weight>
#<Height>176</Height>
#<Smoker>1</Smoker>
#<Notice></Notice>
#<Object>a_circ_p</Object>
#<Object_Index>1</Object_Index>
#<Pen>Pentrics Alberich1</Pen>
#<Samplerate>1000</Samplerate>
#<Time>[REDACTED]</Time>
#<Date>[REDACTED]</Date>
#<Comment></Comment>
#</meta>


Header ini bonus tak terduga — ada `Samplerate: 1000` eksplisit, plus kovariat demografis (usia, gender, tangan dominan, berat, tinggi, perokok) yang bisa jadi confounder perlu dicatat. Sekarang baris data setelah header:

In [13]:
data_rows = []
for l in lines:
    if l.startswith("#") or not l.strip():
        continue
    data_rows.append([float(x) for x in l.split()])

print(f"Jumlah baris data: {len(data_rows)}")
print(f"Jumlah kolom: {len(data_rows[0])}")
print("5 baris pertama:")
for r in data_rows[:5]:
    print(r)

Jumlah baris data: 10600
Jumlah kolom: 6
5 baris pertama:
[3.01598170227124, 7.42844620892122, 5.9721909807594, 5.92632, 4.68585, 3.91826]
[2.98586877420341, 7.33678388418268, 5.97091783407694, 5.92632, 4.68585, 3.91826]
[2.9856228384788, 7.24345848450073, 5.96962719749549, 5.92632, 4.68585, 3.91826]
[2.99605113173114, 7.13874346287346, 5.96879571629436, 5.92852, 4.67265, 3.92485]
[2.99749112500403, 7.01995905309351, 5.96796133983234, 5.92852, 4.67265, 3.92485]


**Tidak ada nama kolom di berkas maupun di halaman resmi.** 6 kolom numerik tanpa label. Rentang nilainya (~2 sampai ~11) tidak terlihat seperti koordinat piksel digitizer biasa (yang biasanya beratus-ratus/beribu seperti di UCI 395) — lebih mirip unit tersensor/terkalibrasi. Cek pola nilai berulang buat curigai kemungkinan dua sensor beda frekuensi digabung dalam satu tabel:

In [14]:
n_cols = len(data_rows[0])
for c in range(n_cols):
    col = [r[c] for r in data_rows]
    runs, run = [], 1
    for i in range(1, len(col)):
        if col[i] == col[i-1]:
            run += 1
        else:
            runs.append(run); run = 1
    runs.append(run)
    avg_run = sum(runs) / len(runs)
    print(f"kolom {c}: rentang [{min(col):.2f}, {max(col):.2f}]  rata-rata panjang nilai berulang berturut-turut = {avg_run:.2f}")

kolom 0: rentang [2.46, 3.80]  rata-rata panjang nilai berulang berturut-turut = 1.00
kolom 1: rentang [4.91, 7.43]  rata-rata panjang nilai berulang berturut-turut = 1.00
kolom 2: rentang [4.89, 11.20]  rata-rata panjang nilai berulang berturut-turut = 1.00
kolom 3: rentang [4.44, 6.80]  rata-rata panjang nilai berulang berturut-turut = 3.70
kolom 4: rentang [4.06, 5.32]  rata-rata panjang nilai berulang berturut-turut = 3.67
kolom 5: rentang [3.25, 4.60]  rata-rata panjang nilai berulang berturut-turut = 3.67


**Kolom 0-2** berubah tiap baris (~1000 Hz sungguhan, sesuai header). **Kolom 3-5** berulang rata-rata ~3,7 baris berturut-turut — sensor kedua bersampling ~1000/3,7 ≈ 270 Hz, ditahan nilainya (upsampled) supaya panjang tabelnya sama dengan kolom 0-2.

**Status G1 sekarang:** jumlah kolom, tipe data, dan pola dua-sensor-beda-frekuensi sudah terverifikasi — bukan lagi "sepenuhnya tidak diketahui." **Yang masih terbuka:** identitas fisik tiap kolom (mana x, mana y, mana tekanan, mana tilt/akselerasi) — tak ada di berkas maupun halaman resmi. Perlu ditelusuri ke metode paper Pereira dkk. 2016 (rujukan [3] di naskah) atau spesifikasi pen BiSP sebelum kanal-kanal ini bisa dipetakan ke rekayasa kanal di [3.4](../naskah/sempro-skripsi.md).

## 7. Ringkasan verifikasi vs checklist [3.3.3](../naskah/sempro-skripsi.md) dan Lampiran C

In [15]:
status = pd.DataFrame([
    {"Butir": "G1 — struktur kolom NewHandPD", "Status": "Sebagian terjawab", "Catatan": "6 kolom, dua sensor beda frekuensi (~1000Hz & ~270Hz). Identitas fisik kolom belum diketahui — cek paper Pereira 2016."},
    {"Butir": "G2 — frekuensi sampling UCI 395", "Status": "Terjawab", "Catatan": "~142,86 Hz dari selisih timestamp."},
    {"Butir": "G2 — frekuensi sampling NewHandPD", "Status": "Terjawab", "Catatan": "1000 Hz dinyatakan eksplisit di header berkas (kolom 0-2); kolom 3-5 efektif ~270Hz."},
    {"Butir": "G6 — panjang urutan per tugas", "Status": "Terjawab", "Catatan": "Test ID 2 median 13,1s (~1800 baris) — cukup panjang, bukan risiko utama."},
    {"Butir": "G7 — kelengkapan tugas per subjek (UCI 395)", "Status": "Terjawab sebagian", "Catatan": "Cuma 75% subjek (30/40) punya rekaman Test ID 2."},
    {"Butir": "Jumlah subjek 62 PD + 15 HC", "Status": "Hampir cocok, belum pasti", "Catatan": "hw_dataset+new_dataset = 63 PD, 16 HC. Selisih 1 di tiap sisi, perlu rekonsiliasi manual."},
    {"Butir": "TEMUAN BARU — definisi Test ID 2 / \"STCP\"", "Status": "TIDAK COCOK NASKAH", "Catatan": "Readme resmi: 'Circular Motion Test, subjects draw circles around the red point' — bukan pena diam tanpa gerakan volunter. Premis inti analisis primer perlu ditinjau ulang."},
])
status

,Butir,Status,Catatan
0,G1 — struktur kolom NewHandPD,Sebagian terjawab,"6 kolom, dua sensor beda frekuensi (~1000Hz & ..."
1,G2 — frekuensi sampling UCI 395,Terjawab,"~142,86 Hz dari selisih timestamp."
2,G2 — frekuensi sampling NewHandPD,Terjawab,1000 Hz dinyatakan eksplisit di header berkas ...
3,G6 — panjang urutan per tugas,Terjawab,"Test ID 2 median 13,1s (~1800 baris) — cukup p..."
4,G7 — kelengkapan tugas per subjek (UCI 395),Terjawab sebagian,Cuma 75% subjek (30/40) punya rekaman Test ID 2.
5,Jumlah subjek 62 PD + 15 HC,"Hampir cocok, belum pasti","hw_dataset+new_dataset = 63 PD, 16 HC. Selisih..."
6,"TEMUAN BARU — definisi Test ID 2 / ""STCP""",TIDAK COCOK NASKAH,"Readme resmi: 'Circular Motion Test, subjects ..."


## 8. Rekomendasi tindak lanjut

1. **Prioritas tertinggi, sebelum Tahap 2:** putuskan bagaimana menangani temuan Test ID 2. Opsi yang tersedia — cek paper asli Isenkul/Sakar (rujukan [16]) untuk lihat apakah mereka tetap menyebutnya "Stability Test on Certain Point" dengan definisi operasional yang berbeda dari readme UCI (kemungkinan real: tugas ini mengukur stabilitas *saat* bergerak melingkar di sekitar titik, bukan diam total) — atau revisi [3.3.1](../naskah/sempro-skripsi.md), [3.6.4](../naskah/sempro-skripsi.md), dan [3.6.7](../naskah/sempro-skripsi.md) supaya konsisten dengan definisi resmi.
2. Rekonsiliasi selisih jumlah subjek (63 vs 62 PD, 16 vs 15 HC) — cek apakah ada berkas duplikat atau subjek yang perlu dikeluarkan.
3. Telusuri paper Pereira dkk. 2016 buat identitas kolom NewHandPD (G1 sisanya) sebelum menulis rekayasa kanal di Tahap 2.
4. Unduh set `PatientSignal.zip` juga (baru set Healthy yang diunduh di sini) sebelum Tahap 2 dimulai sungguhan.
5. Periksa metadata UCI 395 buat label keparahan (G4) — belum dicek di notebook ini.